<a href="https://colab.research.google.com/github/edik06031-rgb/DTA_2026/blob/main/project_store_sales_analysis/store_sales_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Аналіз продажу магазину

Вихідний дата сет

In [40]:
# Запусти цей блок, щоб отримати датасет
import pandas as pd
import numpy as np

np.random.seed(42)
n = 300


categories = ['Молочне', 'Хліб та випічка', 'М\'ясо та риба', 'Овочі та фрукти', 'Напої']
products = {
    'Молочне':          ['Молоко 1л', 'Кефір 0.5л', 'Сир кисломолочний', 'Масло вершкове', 'Йогурт'],
    'Хліб та випічка':  ['Хліб білий', 'Батон', 'Булочка', 'Хліб житній', 'Круасан'],
    'М\'ясо та риба':   ['Куряче філе', 'Свинина', 'Ковбаса', 'Риба хек', 'Сосиски'],
    'Овочі та фрукти':  ['Картопля', 'Морква', 'Яблука', 'Банани', 'Помідори'],
    'Напої':            ['Вода 1.5л', 'Сік апельсиновий', 'Чай', 'Кава мелена', 'Лимонад'],
}
prices = {
    'Молоко 1л': 38, 'Кефір 0.5л': 22, 'Сир кисломолочний': 55, 'Масло вершкове': 95, 'Йогурт': 30,
    'Хліб білий': 28, 'Батон': 25, 'Булочка': 12, 'Хліб житній': 32, 'Круасан': 18,
    'Куряче філе': 165, 'Свинина': 210, 'Ковбаса': 145, 'Риба хек': 120, 'Сосиски': 98,
    'Картопля': 20, 'Морква': 15, 'Яблука': 45, 'Банани': 52, 'Помідори': 60,
    'Вода 1.5л': 18, 'Сік апельсиновий': 68, 'Чай': 85, 'Кава мелена': 155, 'Лимонад': 35,
}

chosen_cats = np.random.choice(categories, n, p=[0.25, 0.20, 0.20, 0.20, 0.15])
chosen_products = [np.random.choice(products[c]) for c in chosen_cats]

df_origin = pd.DataFrame({
    'transaction_id': range(1001, 1001 + n),
    'date':           pd.to_datetime(
                          np.random.choice(pd.date_range('2024-03-01', '2024-03-31'), n)
                      ),
    'hour':           np.random.choice(range(8, 21), n, p=[0.04,0.06,0.08,0.10,0.12,0.12,0.11,0.10,0.09,0.08,0.06,0.04,0.00]),
    'category':       chosen_cats,
    'product':        chosen_products,
    'quantity':       np.random.randint(1, 6, n),
    'price_uah':      [prices[p] for p in chosen_products],
    'customer_age':   np.random.randint(18, 70, n),
    'payment':        np.random.choice(['Готівка', 'Картка', 'Телефон'], n, p=[0.35, 0.50, 0.15]),
})


In [9]:
df = df_origin.copy()

In [29]:
df

,transaction_id,date,hour,category,product,quantity,price_uah,customer_age,payment,total_uah
0,1001,2024-03-17,17,Хліб та випічка,Круасан,4,18,28,Готівка,72
1,1002,2024-03-13,14,Напої,Кава мелена,3,155,40,Картка,465
2,1003,2024-03-01,18,Овочі та фрукти,Яблука,1,45,43,Картка,45
3,1004,2024-03-02,8,М'ясо та риба,Риба хек,3,120,51,Готівка,360
4,1005,2024-03-09,19,Молочне,Йогурт,5,30,63,Картка,150
...,...,...,...,...,...,...,...,...,...,...
295,1296,2024-03-08,10,М'ясо та риба,Сосиски,4,98,48,Готівка,392
296,1297,2024-03-05,13,Овочі та фрукти,Морква,5,15,28,Телефон,75
297,1298,2024-03-04,11,Молочне,Молоко 1л,3,38,27,Картка,114
298,1299,2024-03-24,14,М'ясо та риба,Ковбаса,1,145,20,Готівка,145


Блок 1 - Перше знайомство з даними

In [27]:
df.head(7)

,transaction_id,date,hour,category,product,quantity,price_uah,customer_age,payment,total_uah
0,1001,2024-03-17,17,Хліб та випічка,Круасан,4,18,28,Готівка,72
1,1002,2024-03-13,14,Напої,Кава мелена,3,155,40,Картка,465
2,1003,2024-03-01,18,Овочі та фрукти,Яблука,1,45,43,Картка,45
3,1004,2024-03-02,8,М'ясо та риба,Риба хек,3,120,51,Готівка,360
4,1005,2024-03-09,19,Молочне,Йогурт,5,30,63,Картка,150
5,1006,2024-03-03,13,Молочне,Кефір 0.5л,3,22,51,Картка,66
6,1007,2024-03-31,15,Молочне,Масло вершкове,2,95,46,Телефон,190


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   transaction_id  300 non-null    int64         
 1   date            300 non-null    datetime64[ns]
 2   hour            300 non-null    int64         
 3   category        300 non-null    object        
 4   product         300 non-null    object        
 5   quantity        300 non-null    int64         
 6   price_uah       300 non-null    int64         
 7   customer_age    300 non-null    int64         
 8   payment         300 non-null    object        
 9   total_uah       300 non-null    int64         
dtypes: datetime64[ns](1), int64(6), object(3)
memory usage: 23.6+ KB


In [26]:
df["total_uah"] = df.quantity * df.price_uah

In [14]:
df.dtypes

,0
transaction_id,int64
date,datetime64[ns]
hour,int64
category,object
product,object
quantity,int64
price_uah,int64
customer_age,int64
payment,object


In [15]:
print(f"Size:{df.shape[0]} column x {df.shape[1]} rows")

Size:300 column x 9 rows


In [16]:
print("Types of Data")
display(df.dtypes)

Types of Data


,0
transaction_id,int64
date,datetime64[ns]
hour,int64
category,object
product,object
quantity,int64
price_uah,int64
customer_age,int64
payment,object


4.Перевір, чи є пропущені значення.

In [20]:
#df.isna()
#df.isnull()
df.isnull().sum()

,0
transaction_id,0
date,0
hour,0
category,0
product,0
quantity,0
price_uah,0
customer_age,0
payment,0


In [ ]:
print(f"Пропущенні значення по стовбцам :\n\n{}")

5.Виведи основну статистику (describe) для числових стовпців.

In [31]:
print("Статистика: \n")
df.describe().round(2)

Статистика: 



,transaction_id,date,hour,quantity,price_uah,customer_age,total_uah
count,300.00,300,300.00,300.00,300.00,300.0,300.00
mean,1150.50,2024-03-16 02:14:24,13.72,2.97,63.45,43.0,186.97
min,1001.00,2024-03-01 00:00:00,8.00,1.00,12.00,18.0,12.00
25%,1075.75,2024-03-08 00:00:00,12.00,2.00,22.00,31.0,60.00
50%,1150.50,2024-03-16 12:00:00,14.00,3.00,38.00,42.5,120.00
75%,1225.25,2024-03-24 00:00:00,16.00,4.00,95.75,55.0,256.25
max,1300.00,2024-03-31 00:00:00,19.00,5.00,210.00,69.0,840.00
std,86.75,NaN,2.74,1.44,51.68,14.6,180.27


ВИСНОВОК:

Дата сет містить інформацію про 300 транзакцій, 10 стовпців,пропущених значень немає.
Середня сума чеку - 186,97
Мінімальний чек - 12.00 грн().
Максимальний чек - 840.00 грн().
Підозрілих даних не виявлено

```
[1, 7, 9, 10, 12, 15]
```
Медіана: (9 + 10) / 2 = 9,5
```
[1, 7, 9, 10, 12, 15, 20]
```
Медіана: 10
---
```
[1, 7, 9, 10, 12, 15, 20]
```
Медіана: 10 - `50%`  
`min` = 1  
`max` = 20  
`count` = 7  
`mean` = 10.57  
`25%` ->  
0% - min -> 25% -> 50% - median -> 75% -> 100% - max
std ->  
10.57 - 1 = 9.57
10.57 - 7 = 3.57
..
20 - 10.57 = 9.43

`[1, 1, 1, 1, 1, 6, 7, 8, 9, 10]`
1% - min -> 1  
25% -> 1  
50% - median -> 3.5  
75% -> 8  
100% - max -> 10

In [24]:
s = [1, 7, 9, 10, 12, 15, 20]
mean = sum(s)/len(s)
mean

10.571428571428571

In [25]:
s = [1, 1, 1, 1, 1, 1, 1000]
mean = sum(s)/len(s)
mean
# std -> 143.71 - 1 = 142.71
mediana = 1

БЛОК2 -Фільтрація та відбір



1. Відбери всі покупки

In [34]:
df[df["category"] == "Мясо та риба"].head()

,transaction_id,date,hour,category,product,quantity,price_uah,customer_age,payment,total_uah


In [35]:
meat_fish = df[df["category"] == "Мясо та риба"]
print(f(''))

2.Знайди транзакції, де сума (total_uah) перевищує 500 грн.


In [36]:
big = df[df['total_uah'] > 500]
print(f'Покупки > 500 грн: {len(big)}\n')
display(big[['product', 'quantity', 'total_uah']].head())
#print(big[['product', 'quantity', 'total_uah']].head())

Покупки > 500 грн: 17



,product,quantity,total_uah
24,Свинина,4,840
28,Куряче філе,4,660
81,Ковбаса,4,580
89,Риба хек,5,600
93,Свинина,3,630


відсоток великих покупок

In [ ]:
pct_big = len(big) / len(df) * 100
# round(pct_big, 1)
print(f"Частка покупок > 500: {len(big)}")

In [47]:
for cat in big.category.unique():
    print(f"{cat} - {big[big["category"] == cat].count()}")
    # print(f"{cat}")

М'ясо та риба - transaction_id    14
date              14
hour              14
category          14
product           14
quantity          14
price_uah         14
customer_age      14
payment           14
total_uah         14
dtype: int64
Напої - transaction_id    3
date              3
hour              3
category          3
product           3
quantity          3
price_uah         3
customer_age      3
payment           3
total_uah         3
dtype: int64


3.Відбери покупки, зроблені карткою (payment == 'Картка'), кількістю більше 2 одиниць.

In [39]:
df["payment"] == " Картка"
card_mult = df[(df["payment"]== "Картка") & (df["quantity"] > 2)]
print("Картка + кількість > 2")

Знайди всі покупки покупців молодше 25 років.

In [46]:
young = df[df["customer_age"] < 25]
print("Покупців до 25 : ", len(young))

Покупців до 25 :  36


5.Скільки унікальних товарів є в датасеті?

In [48]:
n_unique = df["product"].nunique()
#n_unique

pr_unique = df["product"].unique()
#pr_unique

print("Скільки унікальних товарів :", n_unique, "\n")
print("перелік унікальних товарів :\n",pr_unique)


Скільки унікальних товарів : 25 

перелік унікальних товарів :
 [np.str_('Круасан') np.str_('Кава мелена') np.str_('Яблука')
 np.str_('Риба хек') np.str_('Йогурт') np.str_('Кефір 0.5л')
 np.str_('Масло вершкове') np.str_('Сік апельсиновий') np.str_('Ковбаса')
 np.str_('Картопля') np.str_('Сир кисломолочний') np.str_('Морква')
 np.str_('Куряче філе') np.str_('Хліб житній') np.str_('Батон')
 np.str_('Хліб білий') np.str_('Свинина') np.str_('Сосиски')
 np.str_('Молоко 1л') np.str_('Помідори') np.str_('Лимонад')
 np.str_('Булочка') np.str_('Банани') np.str_('Вода 1.5л') np.str_('Чай')]


Висновок

Близіко третини покупок оплачено карткою з яких третина більше 2-х товарів у чеку.
5.7 % транзакцій перевищують чек у розмірі 500 грн, здебільшого  здійснені у категоріі мясо та риба.
12 % покупців молодшого віку (до 25 років)

Блок 3 - Групування та агрегація

1. підрахувати загальний виторг по кожній категорії. відсортуй за спаданням.

In [51]:
#revenue_by_cat = df.groupby("category")
df.groupby("category")["total_uah"].sum().sort_values(ascending=False)

,total_uah
category,
М'ясо та риба,25223
Напої,10389
Молочне,10075
Овочі та фрукти,6270
Хліб та випічка,4134


2.Знайди топ-5 товарів за кількістю проданих одиниць (сума quantity).

In [57]:
df.groupby("category").agg({"total_uah": "sum","transaction_id": "count","quantity": "sum"}).sort_values(by= ["total_uah","transaction_id"],ascending=False)

,total_uah,transaction_id,quantity
category,,,
М'ясо та риба,25223,61,186
Напої,10389,48,129
Молочне,10075,80,242
Овочі та фрукти,6270,55,158
Хліб та випічка,4134,56,177
